# Cortex-Local 蒸馏训练（Colab）

把 `Eveyz/cortex-rs` 的蒸馏流程搬到 Colab GPU 上跑：先用 Qwen3 老师生成蒸馏数据（`distill_generate.py`），再训 1.58-bit 学生（`distill_train.py`），最后把 `.ctxb` 存到 Google Drive。

## 先看你的 GPU 决定老师（关键）

| Colab GPU | 显存 | 老师怎么选 | 学生训练 |
|---|---|---|---|
| A100 80GB（Pro+） | 80GB | 默认 `Qwen/Qwen3-30B-A3B-Instruct-2507`（bf16 约 60GB），直接跑 | 默认 `--config production`（约 26GB），直接跑 |
| A100 40GB / L4 | 40 / 24GB | 换量化老师 `--teacher Qwen/Qwen3-30B-A3B-AWQ`（约 30GB），或 `--teacher Qwen/Qwen3-8B` | production 勉强（40GB）/ 需降参（24GB：`--batch-size 1 --grad-accum 8 --seq-len 256`） |
| T4（免费） | 16GB | 只能小老师：`--teacher Qwen/Qwen3-4B`（约 8GB），或 AWQ 版 8B | 只能 `--config micro` 做冒烟测试 |

## Colab 的两个坑

- **运行时是临时的**：最长约 12 小时，闲置约 90 分钟会断。所有重要产出（数据、checkpoint、`.ctxb`）都写到 Drive，模型权重放本地磁盘（Drive 太慢，加载不动）。
- **生成是可断点续跑的**：`distill_generate.py` 会数已有输出文件里每个类别已有的条数，只补剩下的。断了重跑同一条命令就行。

跑之前先在下面把 `TEACHER` 改成你那张卡能装的版本。

In [ ]:
# 1. 看 GPU 和磁盘够不够（老师 bf16 约 60GB，磁盘至少留 80GB 空闲）
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
!df -h /content | tail -1

In [ ]:
# 2. 挂 Drive：产出都放这里，运行时断了也不丢
from google.colab import drive
drive.mount('/content/drive')

import os
WORK = '/content/drive/MyDrive/cortex-local'
os.makedirs(WORK, exist_ok=True)
print('work dir:', WORK)

In [ ]:
# 3. 拉代码（已是 GitHub main，d1e8497 之后的更新直接 git pull）
%cd /content
!test -d cortex-rs || git clone https://github.com/Eveyz/cortex-rs.git
%cd /content/cortex-rs
!git pull --ff-only
!git log --oneline -3

In [ ]:
# 4. 装依赖：vLLM（带老师推理）+ hf_transfer（60GB 权重下载加速）
# 注意：这步会装/换 torch，约几分钟；装完按提示重启运行时再继续
%pip install -q vllm hf_transfer

import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
# 权重缓存放本地磁盘，别放 Drive（太慢）
os.environ['HF_HOME'] = '/content/.cache/huggingface'

!python -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"
!python -c "import vllm; print('vllm', vllm.__version__)"

In [ ]:
# 5. 冒烟测试：只生成 20 条，看老师能不能跑起来、格式对不对
# A100 80GB 用默认老师；小卡把 TEACHER 改成上面表格里的版本
TEACHER = 'Qwen/Qwen3-30B-A3B-Instruct-2507'  # 改这里
WORK = '/content/drive/MyDrive/cortex-local'

!python tools/distill_generate.py --teacher {TEACHER} --out {WORK}/distill-data.jsonl --limit 20

In [ ]:
# 6. 全量生成：约 1.45 万条，30B 老师在 A100 上要几小时
# 用 nohup 放后台跑，浏览器关了/断了也不停；断点续跑直接重跑同一条命令
TEACHER = 'Qwen/Qwen3-30B-A3B-Instruct-2507'  # 和上面保持一致
WORK = '/content/drive/MyDrive/cortex-local'

!nohup python tools/distill_generate.py --teacher {TEACHER} --out {WORK}/distill-data.jsonl > {WORK}/gen.log 2>&1 &
print('started in background, log:', WORK + '/gen.log')

In [ ]:
# 7. 看生成进度（随时可跑这个 cell）
WORK = '/content/drive/MyDrive/cortex-local'
!tail -5 {WORK}/gen.log
!wc -l {WORK}/distill-data.jsonl 2>/dev/null || echo 'no output yet'

In [ ]:
# 8. 训学生：production 配置 2 epochs，约 26GB 显存
# 24GB 卡降参：--batch-size 1 --grad-accum 8 --seq-len 256；T4 只跑 --config micro 冒烟
WORK = '/content/drive/MyDrive/cortex-local'

!python tools/distill_train.py --data {WORK}/distill-data.jsonl --config production --epochs 2 --out {WORK}/student.ctxb --ckpt-dir {WORK}/ckpts

In [ ]:
# 9. 验收：确认 .ctxb 在 Drive 里，并对一下 PyTorch/Rust 的 logit 一致性
WORK = '/content/drive/MyDrive/cortex-local'

!ls -lh {WORK}/student.ctxb {WORK}/ckpts/ 2>/dev/null
# 自检（需要先 cargo build 出 cortex-cli；Colab 上可装 rust 再编，或拿回本地跑）
# !python tools/train_seed_qat.py --self-check --rust-bin ./target/release/cortex-cli

## 跑完之后

- `student.ctxb` 在 Drive 的 `cortex-local/` 里，拿回本地用 `cortex-cli` 就能加载生成。
- 想在 Colab 上也跑 Rust 自检：`curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh`，再 `cargo build --release -p cortex-runtime`，然后取消上面最后那个 cell 的注释。
- 已知待修：`--joint-hippocampus` 的自检目前误差 2.610e-03，超了 1e-3 门槛——正式 joint 训练前要先定位，别直接上。